# Chapter 5, Exercise 5: Subword versus morphological tokenization of وسيكتبونها

> **How to use this notebook.** Open it in Google Colab (File > Upload notebook, or the Colab badge on the companion website), then run the cells from top to bottom. Runtime > Change runtime type lets you pick a GPU when one is recommended below. Everything else runs on the free CPU tier.

## The exercise

**Chapter 5, Exercise 5.** Choose an output unit for an end-to-end recognizer for (a) MSA broadcast news and (b) spontaneous dialectal speech with no standard spelling, using Table 5.3. Justify each, referring to OOV risk and morphology. Then, using Python with a subword library such as SentencePiece [11] or a morphological analyzer such as CAMeL Tools [12], tokenize وسيكتبونها (wa-sa-yaktubūnahā, 'and they will write it') with each method and compare the actual output tokens.

## Requirements

No GPU is required; the notebook runs on Colab's free CPU runtime.


<a href="https://colab.research.google.com/github/arabic-speech-book/arabic-speech-book.github.io/blob/main/docs/solutions/Chapter_05_Exercise_05.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

## 1. Choosing the output unit (see the solution file for the full argument)

* **(a) MSA broadcast news:** learned subwords (BPE or unigram, vocabulary of a few thousand) trained on normalized broadcast transcripts, optionally with an external MSA language model. Large, consistent, orthographically standard text is exactly what a subword tokenizer exploits, and word-level OOVs from morphology disappear.
* **(b) Spontaneous dialect with no standard spelling:** characters, or a *small* subword vocabulary with full character coverage (which behaves close to characters), trained on transcripts written under one stated convention (CODA). Characters keep the inventory fixed while spelling varies, and no dialect-wide morphological analyzer is available to justify morpheme units.

## 2. Train subword tokenizers with SentencePiece

We need some Arabic text to train on. The notebook downloads the FLEURS Arabic (`ar_eg`) train transcripts from the Hugging Face Hub (about 1.5 MB, CC BY 4.0); any Arabic text file works. Three vocabulary sizes are trained so that the effect on the segmentation is visible.

In [1]:
!pip install -q sentencepiece camel-tools huggingface_hub pandas
!camel_data -i disambig-mle-calima-msa-r13

No new packages will be installed.


In [2]:
import re, unicodedata, pandas as pd, sentencepiece as spm
from huggingface_hub import hf_hub_download

tsv = hf_hub_download("google/fleurs", "data/ar_eg/train.tsv", repo_type="dataset")
df = pd.read_csv(tsv, sep="\t", header=None, quoting=3)
# column 2 = raw transcription (see FLEURS README); apply the same light normalization as scoring would
DIAC = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
def norm(t):
    t = unicodedata.normalize("NFC", str(t)); t = DIAC.sub("", t).replace("\u0640", "")
    t = re.sub(r"[\u060C\u061B\u061F!-/:-@\[-`{-~]", " ", t)
    return re.sub(r"\s+", " ", t).strip()
sents = [norm(s) for s in df[2].tolist()]
with open("ar_train.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(sents))
print(len(sents), "training sentences;", sum(len(s.split()) for s in sents), "words")

WORD = "وسيكتبونها"
print("\nDoes the exact word occur in the training text?", any(WORD in s.split() for s in sents))

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2104 training sentences; 38272 words

Does the exact word occur in the training text? False


In [3]:
results = {}
for model_type in ["bpe", "unigram"]:
    for vocab in [500, 2000, 8000]:
        prefix = f"sp_{model_type}_{vocab}"
        spm.SentencePieceTrainer.train(input="ar_train.txt", model_prefix=prefix, vocab_size=vocab,
                                       model_type=model_type, character_coverage=1.0,
                                       bos_id=-1, eos_id=-1, minloglevel=2)
        sp = spm.SentencePieceProcessor(model_file=f"{prefix}.model")
        pieces = sp.encode(WORD, out_type=str)
        results[(model_type, vocab)] = pieces
        print(f"{model_type:7s} vocab={vocab:5d}: {' + '.join(pieces)}   ({len(pieces)} pieces)")

bpe     vocab=  500: ▁و + سي + كت + ب + ون + ها   (6 pieces)
bpe     vocab= 2000: ▁و + سي + كت + ب + ون + ها   (6 pieces)
bpe     vocab= 8000: ▁وسي + كت + ب + ونها   (4 pieces)


unigram vocab=  500: ▁و + س + ي + ك + ت + ب + ون + ها   (8 pieces)


unigram vocab= 2000: ▁و + س + ي + ك + ت + ب + ون + ها   (8 pieces)


unigram vocab= 8000: ▁وس + يك + ت + ب + ونها   (5 pieces)


The leading `▁` marks a word boundary in SentencePiece. Notice how the segmentation **changes with the vocabulary size and the algorithm**: this is the reason the book (Table 4.4 note, Exercise 2.5) refuses to promise one "correct" BPE output. Whatever the size, no piece is out of vocabulary, because `character_coverage=1.0` guarantees a character fallback.

## 3. Segment the same word with a morphological analyzer (CAMeL Tools)

In [4]:
import camel_tools
from camel_tools.disambig.mle import MLEDisambiguator
from camel_tools.tokenizers.morphological import MorphologicalTokenizer

mle = MLEDisambiguator.pretrained("calima-msa-r13")
print("CAMeL Tools", camel_tools.__version__, "| MLE disambiguator, calima-msa-r13")
for scheme in ["atbtok", "d3tok", "bwtok"]:
    tok = MorphologicalTokenizer(disambiguator=mle, scheme=scheme, split=True)
    print(f"scheme={scheme:7s}:", " ".join(tok.tokenize([WORD])))
a = mle.disambiguate([WORD])[0].analyses[0].analysis
print("\nchosen analysis:", a["diac"], "|", a["bw"])
print("lemma:", a["lex"], "| pos:", a["pos"], "| gloss:", a["gloss"])

CAMeL Tools 1.6.0 | MLE disambiguator, calima-msa-r13
scheme=atbtok : و+ س+ يكتبون +ها
scheme=d3tok  : و+ س+ يكتبون +ها
scheme=bwtok  : و+ س+ ي+ كتب +ون +ها

chosen analysis: وَسَيَكْتُبُونَها | وَ/PART+سَ/FUT_PART+يَ/IV3MP+كْتُب/IV+ُونَ/IVSUFF_SUBJ:MP_MOOD:I+ها/IVSUFF_DO:3FS
lemma: كَتَب | pos: verb | gloss: [part.]_+_will_+_they_(people)+write+it;them;her


## 4. Compare

| Method | Output for وسيكتبونها | Pieces | What the pieces are |
|---|---|---|---|
| Manual morphological analysis | وَ + سَ + يَكْتُبُونَ + ها | 4 | conjunction, future marker, inflected verb, object pronoun |
| CAMeL Tools D3 / ATB (`d3tok`, `atbtok`) | و+ س+ يكتبون +ها | 4 | the same four morphemes, labelled |
| CAMeL Tools BW (`bwtok`) | و+ س+ ي+ كتب +ون +ها | 6 | also splits the subject prefix and suffix around the root stem كتب |
| SentencePiece BPE / unigram (trained on 2,104 FLEURS sentences) | see the printed table; varies with algorithm and vocabulary size (4 to 8 pieces in the tested run, e.g. BPE-8000: ▁وسي + كت + ب + ونها) | 4 to 8 | frequent character strings; some coincide with morphemes (a leading و, a trailing ها or ون), others cut through the root (كت + ب) |

Both methods remove the word-level OOV problem for this word, which is absent from the training text. The analyzer's pieces carry meaning and are stable across contexts; the subword pieces are whatever was frequent in the training text, and they change when the text or the vocabulary changes. For MSA with a good analyzer either is usable; for a dialect without one, only the subword route (or characters) is available.